In [ ]:
import cv2
import re
import numpy as np
from ultralytics import YOLO
from paddleocr import PaddleOCR
from PIL import Image, ImageDraw, ImageFont
from bidi.algorithm import get_display
from collections import Counter

# =========== Init Models ===========
ocr = PaddleOCR(lang='ar', use_angle_cls=True)
plate_detector = YOLO("best.pt")

# =========== Arabic Digits Map ===========
ARABIC_DIGITS = {'٠': '0', '١': '1', '٢': '2', '٣': '3', '٤': '4',
                 '٥': '5', '٦': '6', '٧': '7', '٨': '8', '٩': '9'}

def normalize_digits(text):
    # Arabic → English digits
    for ar, en in ARABIC_DIGITS.items():
        text = text.replace(ar, en)
    return text

def draw_arabic_text(frame, text, position,
                     font_size=24, color=(255, 255, 255),
                     bg_color=(0, 100, 0)):
    # Render Arabic text correctly on OpenCV frame
    img_pil = Image.fromarray(frame)
    draw = ImageDraw.Draw(img_pil)
    bidi_text = get_display(text)

    try:
        font = ImageFont.truetype(
            r"D:\Projects\Car plates\arial\ARIAL.TTF", font_size
        )
    except:
        font = ImageFont.load_default()

    bbox = draw.textbbox(position, bidi_text, font=font)
    draw.rectangle(
        [bbox[0]-5, bbox[1]-2, bbox[2]+5, bbox[3]+2],
        fill=bg_color
    )
    draw.text(position, bidi_text, font=font, fill=color)
    return np.array(img_pil)

def extract_plate_info(plate_img):
    # Improve OCR readability
    plate_img = cv2.resize(
        plate_img, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC
    )

    result = ocr.predict(plate_img)
    full_text = ""

    if result:
        for res in result:
            if "rec_texts" in res:
                full_text += " ".join(res["rec_texts"])
            elif isinstance(res, list):
                for line in res:
                    full_text += " " + line[1][0]

    normalized = normalize_digits(full_text)
    nums = "".join(re.findall(r'\d+', normalized))

    if len(nums) > 4 or len(nums) < 2:
        return None

    raw_chars = re.findall(r'[\u0600-\u06FF]', normalized)
    black_list = ['ح', 'ك', 'و', 'ي', 'ة', 'ا']
    chars_list = [c for c in raw_chars if c not in black_list]

    if len(chars_list) > 3:
        chars_list = chars_list[:3]

    if len(nums) >= 2 and len(chars_list) >= 1:
        return f"{' '.join(chars_list)} {nums}".strip()

    return None
# =========== Process input video and generate annotated output ===========
def process_video(input_path, output_path):
    cap = cv2.VideoCapture(input_path)
    w, h = int(cap.get(3)), int(cap.get(4))
    fps = cap.get(cv2.CAP_PROP_FPS)

    writer = cv2.VideoWriter(
        output_path,
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps, (w, h)
    )

    track_history = {}
    frame_id = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame_id += 1
        results = plate_detector.track(
            frame, persist=True, verbose=False
        )[0]

        if results.boxes.id is not None:
            boxes = results.boxes.xyxy.cpu().numpy().astype(int)
            ids = results.boxes.id.cpu().numpy().astype(int)

            for box, obj_id in zip(boxes, ids):
                x1, y1, x2, y2 = box
                # Draw plate bounding box
                cv2.rectangle(
                    frame, (x1, y1), (x2, y2), (0, 255, 0), 2
                )

                plate_crop = frame[y1:y2, x1:x2]
                if plate_crop.size > 0:
                    plate_text = extract_plate_info(plate_crop)
                    if plate_text:
                        track_history.setdefault(
                            obj_id, []
                        ).append(plate_text)
                        
                # Use majority vote for stable OCR result
                if obj_id in track_history:
                    best_guess = Counter(
                        track_history[obj_id]
                    ).most_common(1)[0][0]

                    frame = draw_arabic_text(
                        frame, best_guess, (x1, y1 - 35)
                    )

        writer.write(frame)

    cap.release()
    writer.release()

process_video("Vedio1.mp4", "output_final1.mp4")


c:\Users\thinkpad\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `DISABLE_MODEL_SOURCE_CHECK` to `True`.
C:\Users\thinkpad\AppData\Local\Temp\ipykernel_21544\2620244167.py:10: DeprecationWarning: The parameter `use_angle_cls` has been deprecated and will be removed in the future. Please use `use_textline_orientation` instead.
  ocr = PaddleOCR(lang='ar', use_angle_cls=True)
c:\Users\thinkpad\AppData\Local\Programs\Python\Python313\Lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc

Starting video processing...
Processing frame 50...
Processing frame 100...
Processing frame 150...
Processing frame 200...
Processing frame 250...
Processing frame 300...
Processing frame 350...
Processing frame 400...
Processing frame 450...
Processing frame 500...
Processing frame 550...
Processing frame 600...
Processing frame 650...
Processing frame 700...
Processing frame 750...
Finished processing video: output_final1.mp4
